In [ ]:
# Gerekli Kütüphaneler eklendi.
from textblob import TextBlob
import math
import pandas as pd
import numpy as np
import os
import re
import nltk
from nltk.corpus import stopwords
from pandas import DataFrame
from sklearn.cluster import KMeans
from sklearn import preprocessing
import matplotlib.pyplot as plt
from matplotlib import style
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn import neighbors, datasets
style.use('ggplot')
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 20, 10
from pandas import read_csv
from datetime import datetime
from matplotlib import pyplot
import sklearn
from sklearn.metrics import mean_squared_error
from wordcloud import WordCloud

In [ ]:
# DataSet eklendi
tweet= pd.read_csv('Donald-Tweets!.csv', index_col=None)

In [ ]:
# ilk 5 satır:
tweet.head()

In [ ]:
tweet.columns

In [ ]:
tweet.shape

In [ ]:
tweet.info()

In [ ]:
tweet.isnull().sum()

In [ ]:
tweet.describe()

In [ ]:
# Media_type, Unnamed: 10 ,Unnamed: 11 sütunlarında çok fazla null değer olduğu için silinecek.
# tweet_id, tweet_url bir işe yaramayacağı için silinecek.

In [ ]:
# Media_Type sütununda hangi eşsiz değerler var?
tweet["Media_Type"].unique()

In [ ]:
tweet["Media_Type"].value_counts()

In [ ]:
dropned_data= tweet.drop(['Media_Type','Tweet_Id','Tweet_Url', 'Unnamed: 10', 'Unnamed: 11'], axis=1)

In [ ]:
dropned_data.shape

In [ ]:
# sütunlardaki boş değerler kaç tane?
dropned_data.isnull().sum()

In [ ]:
# kaldırılmış sütunlarla birlikte güncellenmiş DataSet:
dropned_data

In [ ]:
dropned_data.info()

In [ ]:
dropned_data["Type"]

In [ ]:
dropned_data["Type"].unique()

In [ ]:
# Tweet tiplerine baktık ve grafiğe döktük.
type_counts = dropned_data['Type'].value_counts()

# Renkleri belirleyelim
colors = []
for t in type_counts.index:
    if t == 'text':
        colors.append('skyblue')
    elif t == 'link':
        colors.append('orange')
    elif t == 'video':
        colors.append('red')  # video için dikkat çekici renk

# Bar grafiği çiz
plt.bar(type_counts.index, type_counts.values, color=colors)

plt.yscale('log')

plt.title('Tweet Tiplerinin Dağılımı (Log Scale)')
plt.xlabel('Tweet Tipi')
plt.ylabel('Adet (log scale)')
plt.show()


In [ ]:
dropned_data['Datetime'] = pd.to_datetime(dropned_data['Date'] + ' ' + dropned_data['Time'],
                                format='%y-%m-%d %H:%M:%S')

In [ ]:
dropned_data["Datetime"]

In [ ]:
dropned_data.info()

In [ ]:
# ---------------------------------------------
# TWEET TEMİZLEME FONKSİYONU- Noktalamalar var
# ---------------------------------------------
def clean_tweet_N(text):
    text = text.lower()                             # küçült
    text = re.sub(r"http\S+", "", text)             # linkleri sil
    text = re.sub(r"@\w+", "", text)                # mentionları sil
    text = re.sub(r"#", "", text)                   # hashtag işaretini sil
    text = re.sub(r"\s+", " ", text).strip()        # boşlukları düzenle
    return text



In [ ]:
dropned_data["Clean_Text_N"] = dropned_data["Tweet_Text"].astype(str).apply(clean_tweet_N)

In [ ]:
# Her hücrenin tipini gör
types = dropned_data["Clean_Text_N"].apply(type)
print(types.value_counts())

In [ ]:
dropned_data.info()

In [ ]:
dropned_data['Clean_Text_N'] = dropned_data['Clean_Text_N'].astype('string')

In [ ]:
dropned_data.info()

In [ ]:
# ZAMAN SERİSİ ANALİZLERİ 

In [ ]:
# Saat bazında tweet sayısı
hourly = dropned_data.groupby(dropned_data['Datetime'].dt.hour).size()

# Grafiği çiz
hourly.plot(kind='bar', figsize=(12,6), title='Saat Bazında Tweet Sayısı')
plt.xlabel('Saat')
plt.ylabel('Tweet Sayısı')
plt.show()


Silinsin mi?

In [ ]:
# Pie chart çizimi-  Saat bazında tweet dağılımı
fig, ax = plt.subplots(figsize=(10, 10))
shap = hourly.values
labels = hourly.index.values

ax.pie(shap, labels=labels, startangle=90, shadow=False)
plt.title('Saat Bazında Tweet Dağılımı')
plt.show()

In [ ]:
# Günlük tweet sayısı:
daily = tweet.groupby(dropned_data['Datetime'].dt.date).size()
daily.plot(figsize=(14,6), title='Günlük Tweet Sayısı')
plt.xlabel('Tarih')
plt.ylabel('Tweet Sayısı')
plt.show()

In [ ]:
# Haftalık tweet sayısı
import plotly.express as px

dropned_data['Week'] = dropned_data['Datetime'].dt.to_period('W').apply(lambda r: r.start_time)
weekly = dropned_data.groupby('Week').size().reset_index(name='Tweet_Count')
fig_weekly = px.line(weekly, x='Week', y='Tweet_Count', title='Haftalık Tweet Sayısı',
                     labels={'Week':'Hafta','Tweet_Count':'Tweet Sayısı'})
fig_weekly.show()

In [ ]:
# Günlük ortalama polarity
tweet['polarity'] = dropned_data["Clean_Text_N"].apply(lambda x: TextBlob(x).sentiment.polarity)

daily_polarity = tweet.groupby(dropned_data['Datetime'].dt.date)['polarity'].mean()
daily_polarity.plot(figsize=(14,6), title='Günlük Ortalama Polarity')
plt.show()

In [ ]:
# güncellenmiş datadaki bazu bilgiler:
dropned_data.info()

In [ ]:
import plotly.express as px

# Saat bilgisi oluştur (eğer yoksa)
dropned_data['Hour'] = dropned_data['Datetime'].dt.hour

# Saatlik toplam Retweet sayısı
hourly_retweets = dropned_data.groupby('Hour')['Retweets'].sum().reset_index()

# Retweet grafiği
fig_retweet = px.bar(hourly_retweets, x='Hour', y='Retweets',
                     title='Saat Bazlı Retweet Sayısı',
                     labels={'Hour':'Saat','Retweets':'Retweet Sayısı'},
                     color='Retweets')
fig_retweet.show()

# Saatlik toplam Like sayısı
hourly_likes = dropned_data.groupby('Hour')['twt_favourites_IS_THIS_LIKE_QUESTION_MARK'].sum().reset_index()

# Like grafiği
fig_likes = px.bar(hourly_likes, x='Hour', y='twt_favourites_IS_THIS_LIKE_QUESTION_MARK',
                   title='Saat Bazlı Like Sayısı',
                   labels={'Hour':'Saat','twt_favourites_IS_THIS_LIKE_QUESTION_MARK':'Like Sayısı'},
                   color='twt_favourites_IS_THIS_LIKE_QUESTION_MARK')
fig_likes.show()


In [ ]:
dropned_data.head(3)

In [ ]:
# ---------------------------------------------
# TWEET TEMİZLEME FONKSİYONU- Noktalamalar yok.
# ---------------------------------------------
def clean_tweet(text):
    text = text.lower()                             # küçült
    text = re.sub(r"http\S+", "", text)             # linkleri sil
    text = re.sub(r"@\w+", "", text)                # mentionları sil
    text = re.sub(r"#", "", text)                   # hashtag işaretini sil
    text = re.sub(r"[^a-zA-Z0-9ğüşöçıİĞÜŞÖÇ ]", " ", text)  # noktalama sil
    text = re.sub(r"\s+", " ", text).strip()        # boşlukları düzenle
    return text
dropned_data["Clean_Text"] = dropned_data["Tweet_Text"].astype(str).apply(clean_tweet)

In [ ]:
# ---------------------------------------------
# STOPWORDS KALDIRMA (Wordcloud/NLP/LDA için)
# ---------------------------------------------
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    # Not: Veri içinde boş (NaN) değerler varsa hata vermemesi için str(text) ekledim.
    return " ".join([word for word in str(text).split() if word not in stop_words])

# --- DÜZELTME BURADA YAPILDI ---
# clean_text tek başına bir değişken değil, tablonun bir sütunudur.
# Bu yüzden dropned_data["Clean_Text"] şeklinde çağırdık.
dropned_data["NoStop_Clean_Text"] = dropned_data["Clean_Text"].apply(remove_stopwords)

# ---------------------------------------------
# SONU GÖR
# ---------------------------------------------
print(dropned_data[["Tweet_Text", "Clean_Text", "NoStop_Clean_Text"]].head())

In [ ]:
# ---------------------------------------------
# 1️⃣ STEMMING
# ---------------------------------------------
from nltk.stem import SnowballStemmer
stemmer = SnowballStemmer("english")  # İngilizce stemmer
# Türkçe için: SnowballStemmer("turkish")

def stem_text(text):
    return " ".join([stemmer.stem(word) for word in text.split()])

dropned_data["Stemmed_Text"] = dropned_data["NoStop_Clean_Text"].apply(stem_text)

# ---------------------------------------------
# 2️⃣ STEM COMPLETION
# ---------------------------------------------
# Stem → orijinal kelime eşleştirme dictionary'si
stem_dict = {}
for tweet_text in dropned_data['NoStop_Clean_Text']:
    for word in tweet_text.split():
        stem = stemmer.stem(word)
        if stem not in stem_dict:
            stem_dict[stem] = word  # ilk karşılaşılan kelimeyi kaydet

def complete_stem(text):
    return " ".join([stem_dict.get(stemmer.stem(word), word) for word in text.split()])

dropned_data["Stem_Completed_Text"] = dropned_data["Stemmed_Text"].apply(complete_stem)

# ---------------------------------------------
# 3️⃣ WORDCLOUD
# ---------------------------------------------
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = " ".join(dropned_data["Stem_Completed_Text"].values)

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

plt.figure(figsize=(15,7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
# ---------------------------------------------
# 4️⃣ BAG-OF-WORDS / DTM
# ---------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_df=0.9, min_df=5)  # sık geçen ve nadir kelimeleri filtrele
dtm = vectorizer.fit_transform(dropned_data["Stemmed_Text"])

# ---------------------------------------------
# 5️⃣ LDA MODELİ
# ---------------------------------------------
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=5, random_state=42)  # 5 topic örnek
lda.fit(dtm)

# ---------------------------------------------
# 6️⃣ TOPIC’LERİ GÖRÜNTÜLEME
# ---------------------------------------------

def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx+1}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))
        
display_topics(lda, vectorizer.get_feature_names_out(), 10)    

In [ ]:
# Hangi tweet hangi konuya ait
topic_distribution = lda.transform(dtm)
dropned_data['dominant_topic'] = topic_distribution.argmax(axis=1)

# Konu dağılım grafiği
topic_counts = dropned_data['dominant_topic'].value_counts().sort_index()
topic_counts.plot(kind='bar', figsize=(10,5), title="Konu Dağılımı",color='#C6A98A')
plt.xlabel("Konu")
plt.ylabel("Tweet Sayısı")
plt.show()


In [ ]:
# KOnulara göre günük grafik:
topic_trend = dropned_data.groupby([dropned_data['Datetime'].dt.date, 'dominant_topic']).size().unstack().fillna(0)
topic_trend.plot(figsize=(12,6))
plt.title("Topic Trendleri (Günlük)")
plt.show()


In [ ]:
# --------------------------------------
# POLARİTY HESABI
# ----------------------------------------
polarity_tweets = dropned_data["Clean_Text_N"]
storage = []

for i in range(len(polarity_tweets)):
    x = str(polarity_tweets[i])       # i. tweet
    y = TextBlob(x)          # TextBlob ile analiz et
    z = y.sentiment.polarity # Polarity değerini al
    storage.append(z)        # Listeye ekle

# Listeyi DataFrame'e çevir
change = pd.DataFrame({'trend': storage})

# Grafik boyutu ayarlanıyor
rcParams['figure.figsize'] = 20, 10

# Polarity trendini çiz
change.plot.line(title='Tweet Polarity Trend',color='#C8A6A0')
plt.ylabel('Polarity')
plt.xlabel('Tweet Index')
plt.show()

In [ ]:
# --------------------------------------
# SUBJECTIVITY HESABI
# ----------------------------------------
sub_tweets=dropned_data["Clean_Text_N"]
storage2 = []

for i in range(len(sub_tweets)):
    x2 = str(sub_tweets[i])                # i. tweet
    y2 = TextBlob(x2)                  # TextBlob ile analiz et
    z2 = y2.sentiment.subjectivity     # Subjectivity değerini al
    storage2.append(z2)                # Listeye ekle

# Listeyi DataFrame'e çevir
change2 = pd.DataFrame({'trend2': storage2})

# Grafik boyutu ayarlanıyor
rcParams['figure.figsize'] = 20, 10

# Subjectivity trendini çiz
change2.plot.line(title='Tweet Subjectivity Trend',color='#7A8798')
plt.ylabel('Subjectivity')
plt.xlabel('Tweet Index')
plt.show()

In [ ]:
# ----------------------------
# POLARITY HESAPLAMA (İLK 500 TWEET)
# ----------------------------
storage = []

for i in range(500):
    x = str(polarity_tweets[i])        # i. tweet
    y = TextBlob(x)           # TextBlob ile analiz et
    z = y.sentiment.polarity  # Polarity değerini
    storage.append(z)         # Listeye ekle

# Listeyi DataFrame'e çevir
change = pd.DataFrame({'trend': storage})

# Grafik boyutu ayarlanıyor
rcParams['figure.figsize'] = 20, 10

# Polarity trendini çiz
change.plot.line(title='Polarity Trend for First 500 Tweets')
plt.ylabel('Polarity')
plt.xlabel('Tweet Index')
plt.show()

In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

#  Temizlenmiş metni al (stopwords çıkmış, küçük harfli)
texts = dropned_data["NoStop_Clean_Text"].astype(str)

#  Kelimeleri vektörle
vectorizer = CountVectorizer(min_df=10)  # çok nadir kelimeleri çıkar
X = vectorizer.fit_transform(texts)
words = vectorizer.get_feature_names_out()

# Co-occurrence matrisi
co_matrix = (X.T * X).toarray()
np.fill_diagonal(co_matrix, 0)  # kelimenin kendisiyle ilişkisi 0

# En çok bağlantısı olan kelimeleri seç (ör. top 20)
degree_sums = co_matrix.sum(axis=0)
top_indices = degree_sums.argsort()[::-1][:10]  # en yoğun 20 kelime
top_words = words[top_indices]
co_matrix_small = co_matrix[np.ix_(top_indices, top_indices)]

# Ağ oluştur
G = nx.Graph()
for i in range(len(top_words)):
    for j in range(i+1, len(top_words)):
        weight = co_matrix_small[i,j]
        if weight > 0:
            G.add_edge(top_words[i], top_words[j], weight=weight)

# Grafiği çiz
plt.figure(figsize=(12,12))
pos = nx.spring_layout(G, k=0.5)

# Node
nx.draw_networkx_nodes(G, pos, node_size=500, node_color='skyblue')

# Edge
nx.draw_networkx_edges(G, pos, alpha=0.4, width=1)

# Labels
nx.draw_networkx_labels(G, pos, font_size=10)

# Edge ağırlıklarını yaz
edge_labels = nx.get_edge_attributes(G,'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title("Kelime Ağı Analizi (Top 10 Kelime + Ağırlıklar)")
plt.axis("off")
plt.show()


In [ ]:
!pip install gensim


In [ ]:
# ---------------------------------------------
# 7️⃣ LDA / NMF / LSA Karşılaştırması
#     + Coherence (c_v), Topic Diversity
#     + LDA Log-likelihood ve Perplexity
# ---------------------------------------------

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import numpy as np

# 1) Metinleri al (LDA'da kullandığın stem'li ve stopwords'süz metin)
#    Burada dropned_data["Stemmed_Text"] senin zaten oluşturduğun sütun
texts_raw = dropned_data["Stemmed_Text"].astype(str)
texts_tokenized = [text.split() for text in texts_raw]

# 2) Vectorizer ve DTM (sadece bu bölüm için AYRI değişken isimleri)
n_topics = 5  # LDA'da kullandığın konu sayısı ile aynı olmalı

vectorizer_topics = CountVectorizer(max_df=0.9, min_df=5)
dtm_topics = vectorizer_topics.fit_transform(texts_raw)
feature_names_topics = vectorizer_topics.get_feature_names_out()

# 3) Modelleri kur (LDA, NMF, LSA)
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_model.fit(dtm_topics)

nmf_model = NMF(n_components=n_topics, random_state=42, init="nndsvd")
nmf_model.fit(dtm_topics)

lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
lsa_model.fit(dtm_topics)

# 3.1) LDA için log-likelihood ve perplexity (nicel değerlendirme)
log_likelihood = lda_model.score(dtm_topics)
perplexity = lda_model.perplexity(dtm_topics)

print("LDA Log-likelihood:", log_likelihood)
print("LDA Perplexity:", perplexity)
print("=" * 70)

# 4) Bileşenlerden (topic'lerden) top-N kelimeleri çıkaran yardımcı fonksiyon
def get_topics_from_components(components, feature_names, top_n=10):
    topics = []
    for topic in components:
        top_indices = topic.argsort()[:-top_n - 1:-1]
        topics.append([feature_names[i] for i in top_indices])
    return topics

top_n = 10
lda_topics = get_topics_from_components(lda_model.components_, feature_names_topics, top_n=top_n)
nmf_topics = get_topics_from_components(nmf_model.components_, feature_names_topics, top_n=top_n)
lsa_topics = get_topics_from_components(lsa_model.components_, feature_names_topics, top_n=top_n)

# 5) Gensim dictionary (coherence için)
dictionary = Dictionary(texts_tokenized)

def coherence_and_diversity(topics, texts, dictionary):
    # Coherence (c_v)
    coherence_model = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    )
    coherence = coherence_model.get_coherence()

    # Topic Diversity = benzersiz kelime sayısı / toplam kelime sayısı
    all_words = [w for topic in topics for w in topic]
    diversity = len(set(all_words)) / len(all_words)

    return coherence, diversity

# 6) Her model için Coherence ve Topic Diversity hesapla
coh_lda, div_lda = coherence_and_diversity(lda_topics, texts_tokenized, dictionary)
coh_nmf, div_nmf = coherence_and_diversity(nmf_topics, texts_tokenized, dictionary)
coh_lsa, div_lsa = coherence_and_diversity(lsa_topics, texts_tokenized, dictionary)

print("Model Karşılaştırması (Coherence c_v ve Topic Diversity)")
print("-" * 70)
print(f"LDA : Coherence = {coh_lda:.4f}, Topic Diversity = {div_lda:.4f}")
print(f"NMF : Coherence = {coh_nmf:.4f}, Topic Diversity = {div_nmf:.4f}")
print(f"LSA : Coherence = {coh_lsa:.4f}, Topic Diversity = {div_lsa:.4f}")


In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Model": ["LDA", "NMF", "LSA"],
    "Coherence_c_v": [coh_lda, coh_nmf, coh_lsa],
    "Topic_Diversity": [div_lda, div_nmf, div_lsa]
})

print(results_df)


In [ ]:
import matplotlib.pyplot as plt

models = ["LDA", "NMF", "LSA"]
coherences = [coh_lda, coh_nmf, coh_lsa]
diversities = [div_lda, div_nmf, div_lsa]

plt.figure(figsize=(10,4))

# Coherence grafiği
plt.subplot(1, 2, 1)
plt.bar(models, coherences)
plt.title("Modellerin Coherence (c_v) Değerleri")
plt.ylabel("Coherence (c_v)")

# Topic Diversity grafiği
plt.subplot(1, 2, 2)
plt.bar(models, diversities)
plt.title("Modellerin Topic Diversity Değerleri")
plt.ylabel("Topic Diversity")

plt.tight_layout()
plt.show()
